In [62]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

df = pd.read_csv('train.csv')
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [63]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [64]:
y = df.SalePrice
X = df.drop(['SalePrice','Id'], axis=1)

In [65]:
X_train,X_valid,Y_train,Y_valid = train_test_split(X,y,random_state=0)

In [66]:
obj_cols = [col for col in X_train.columns if X_train[col].dtype == 'object' and not X_train[col].isnull().any()]
nums_cols = [col for col in X_train.columns if X_train[col].dtype in ['int64','float64'] and not X_train[col].isnull().any()]

miss_obj_cols = [col for col in X_train.columns if X_train[col].dtype == 'object' and X_train[col].isnull().any()]
miss_nums_cols = [col for col in X_train.columns if X_train[col].dtype in ['int64','float64'] and X_train[col].isnull().any()]

In [67]:
# Apporach with imputation and one-hot encoding

# 1) Imputation of missing values

imputed_X_train = X_train.copy()
imputed_X_valid = X_valid.copy()

from sklearn.impute import SimpleImputer

for col in miss_nums_cols:
    imputed_X_train[col + '_was_missing'] = imputed_X_train[col].isnull()
    imputed_X_valid[col + '_was_missing']  = imputed_X_valid[col].isnull()


imputer1 = SimpleImputer(strategy='mean')
imputed_X_train[miss_nums_cols] = imputer1.fit_transform(imputed_X_train[miss_nums_cols])
imputed_X_valid[miss_nums_cols] = imputer1.transform(imputed_X_valid[miss_nums_cols])



for col in miss_obj_cols:
    imputed_X_train[col + '_was_missing'] = imputed_X_train[col].isnull()
    imputed_X_valid[col + '_was_missing']  = imputed_X_valid[col].isnull()

imputer2 = SimpleImputer(strategy='most_frequent')
imputed_X_train[miss_obj_cols] = imputer2.fit_transform(imputed_X_train[miss_obj_cols])
imputed_X_valid[miss_obj_cols] = imputer2.transform(imputed_X_valid[miss_obj_cols])

# All the missing values are filled


# Only conversion of object into int remains

In [68]:
# 2) One-hot encoding

from sklearn.preprocessing import OneHotEncoder
import pandas as pd

OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Redefine current_obj_cols to include all object type columns in the imputed dataframes

# These are the columns that need one-hot encoding

current_obj_cols = [col for col in imputed_X_train.columns if imputed_X_train[col].dtype == 'object']

# Apply one-hot encoding to these object columns
OH_cols_train = pd.DataFrame(OH_encoder.fit_transform(imputed_X_train[current_obj_cols]))
OH_cols_valid = pd.DataFrame(OH_encoder.transform(imputed_X_valid[current_obj_cols]))

# Set the index to match original frames for proper concatenation
OH_cols_train.index = imputed_X_train.index
OH_cols_valid.index = imputed_X_valid.index

# Get new column names for one-hot encoded features
OH_cols_train.columns = OH_encoder.get_feature_names_out(current_obj_cols)
OH_cols_valid.columns = OH_encoder.get_feature_names_out(current_obj_cols)

# Remove all object columns (both original obj_cols and miss_obj_cols) from the imputed dataframes
# before concatenating with the one-hot encoded versions
num_X_train = imputed_X_train.drop(current_obj_cols, axis=1)
num_X_valid = imputed_X_valid.drop(current_obj_cols, axis=1)

# Concatenate the numerical/boolean columns with the one-hot encoded columns
Final_X_train = pd.concat([num_X_train, OH_cols_train], axis=1)
Final_X_valid = pd.concat([num_X_valid, OH_cols_valid], axis=1)

In [69]:
def scroe(X_train,X_valid,Y_train,Y_valid):
    model = RandomForestRegressor(n_estimators=1000,random_state=0)
    model.fit(X_train,Y_train)
    model_preds = model.predict(X_valid)
    return mean_absolute_error(Y_valid,model_preds)

In [70]:
print("MAE without dropping is :- ")
print(scroe(Final_X_train,Final_X_valid,Y_train,Y_valid))


MAE without dropping is :- 
16893.37499178082


In [ ]:
drop_cols = miss_obj_cols + miss_nums_cols + obj_cols
drop_X_train = X_train.drop(drop_cols, axis=1)
drop_X_valid = X_valid.drop(drop_cols, axis=1)

print("MAE with dropping is :- ")
print(scroe(drop_X_train,drop_X_valid,Y_train,Y_valid))

MAE with dropping is :- 
17130.408789041096


In [ ]:
# final model

final_model = RandomForestRegressor(n_estimators=1000, random_state=0)
final_model.fit(Final_X_train, Y_train)

In [ ]:
dft = pd.read_csv('test.csv')
dft.head()